# LeetCode 796 — Rotate String bằng Python

**Link đề:** https://leetcode.com/problems/rotate-string/
**Độ khó:** Easy · **Chủ đề:** String, String Matching (so khớp chuỗi)

```python
class Solution:
    def rotateString(self, s: str, goal: str) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu phép **xoay chuỗi** (shift/rotation) và mô tả nó bằng **slicing** `s[k:] + s[:k]`.
2. Viết cách mô phỏng từng lần xoay (baseline `O(n²)`).
3. Nắm **insight kinh điển**: `goal` là một phép xoay của `s` ⇔ `len(s) == len(goal)` **và** `goal in s + s`.
4. Biết thêm cách so khớp tuyến tính **KMP** để bảo đảm `O(n)` trong trường hợp xấu nhất.
5. Trace, chứng minh, phân tích Time/Space, luyện biến thể.

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Cho hai chuỗi `s` và `goal`. Trả `True` nếu và chỉ nếu `s` có thể trở thành `goal` sau **một số** lần shift.

**Một lần shift**: lấy ký tự **trái nhất** của `s` đưa xuống **cuối**. Ví dụ `"abcde"` → `"bcdea"`.

| s | goal | Output | Giải thích |
|---|---|:---:|---|
| `"abcde"` | `"cdeab"` | `True` | shift 2 lần: `abcde → bcdea → cdeab` |
| `"abcde"` | `"abced"` | `False` | thứ tự tương đối `d, e` bị đảo, không xoay nào tạo ra |

**Constraints:** `1 <= s.length, goal.length <= 100`; chỉ chữ thường. Lưu ý: độ dài hai chuỗi **có thể khác nhau**!

"Một số lần" bao gồm cả **0 lần** → `s == goal` thì `True`.

In [1]:
s = "abcde"
cur = s
for k in range(len(s) + 1):
    print(f"sau {k} lần shift: {cur}")
    cur = cur[1:] + cur[0]

sau 0 lần shift: abcde
sau 1 lần shift: bcdea
sau 2 lần shift: cdeab
sau 3 lần shift: deabc
sau 4 lần shift: eabcd
sau 5 lần shift: abcde


Quan sát: sau `len(s)` lần shift ta quay về chuỗi ban đầu → chỉ có **tối đa `n` phép xoay khác nhau** (k = 0..n-1).

## 2. Nền tảng Python: indexing và slicing chuỗi

- `s[0]` ký tự đầu; `s[-1]` ký tự cuối.
- `s[a:b]` lấy từ vị trí `a` đến **trước** `b`. Bỏ `a` = từ đầu, bỏ `b` = tới hết.
- `s[1:]` = bỏ ký tự đầu; `s[:1]` = chỉ ký tự đầu.
- `+` nối chuỗi. Chuỗi Python **bất biến** (immutable): mọi phép cắt/nối tạo **chuỗi mới**.

In [2]:
s = "abcde"
print("s[0] =", s[0], "| s[-1] =", s[-1])
print("s[1:] =", s[1:], "| s[:1] =", s[:1])
print("s[2:] =", s[2:], "| s[:2] =", s[:2])
print("s[2:] + s[:2] =", s[2:] + s[:2])
try:
    s[0] = "z"
except TypeError as e:
    print("Không sửa tại chỗ được:", e)

s[0] = a | s[-1] = e
s[1:] = bcde | s[:1] = a
s[2:] = cde | s[:2] = ab
s[2:] + s[:2] = cdeab
Không sửa tại chỗ được: 'str' object does not support item assignment


### 2.1. Công thức xoay `k` lần: `s[k:] + s[:k]`

Shift `k` lần = cắt `k` ký tự đầu, dán xuống cuối. Không cần lặp `k` lần.

Ví dụ `s = "abcde"`, `k = 2`: `s[2:] = "cde"`, `s[:2] = "ab"` → `"cdeab"`.

In [3]:
def rotate(s: str, k: int) -> str:
    return s[k:] + s[:k]

for k in range(6):
    print(k, rotate("abcde", k))

0 abcde
1 bcdea
2 cdeab
3 deabc
4 eabcd
5 abcde


## 3. Cách 1 — Mô phỏng từng lần shift (baseline)

1. Nếu độ dài khác nhau → `False` ngay.
2. Lặp `n` lần: nếu `cur == goal` → `True`; nếu không, shift một lần.
3. Hết vòng → `False`.

In [4]:
def rotate_string_simulate(s: str, goal: str) -> bool:
    if len(s) != len(goal):
        return False
    cur = s
    for _ in range(len(s)):
        if cur == goal:
            return True
        cur = cur[1:] + cur[0]
    return False

print(rotate_string_simulate("abcde", "cdeab"))  # True
print(rotate_string_simulate("abcde", "abced"))  # False

True
False


`for _ in range(n)`: dấu gạch dưới `_` là tên biến quy ước "không dùng tới", chỉ cần lặp đúng `n` lần.

**Độ phức tạp:** mỗi vòng tạo chuỗi mới `O(n)` và so sánh `O(n)`; `n` vòng → `O(n²)` thời gian, `O(n)` bộ nhớ.

## 4. Cách 2 — Thử mọi `k` bằng slicing (vẫn `O(n²)` nhưng gọn)

In [5]:
def rotate_string_all_k(s: str, goal: str) -> bool:
    return len(s) == len(goal) and any(s[k:] + s[:k] == goal for k in range(len(s)))

print(rotate_string_all_k("abcde", "cdeab"), rotate_string_all_k("abcde", "abced"))

True False


### `any(...)` và đánh giá ngắn mạch `and`

- `any(iterable)` trả `True` nếu **có ít nhất một** phần tử đúng, dừng ngay khi gặp.
- `A and B`: nếu `A` sai thì **không tính** `B` (short-circuit). Nhờ vậy độ dài khác nhau thì không mất công thử các `k`.

In [6]:
print(any([False, False, True]), any([]))
def noisy():
    print("  (B đã được tính)")
    return True
print("False and noisy() ->", False and noisy())
print("True and noisy()  ->", True and noisy())

True False
False and noisy() -> False
  (B đã được tính)
True and noisy()  -> True


## 5. Nút thắt và insight — nối `s` với chính nó

Nút thắt: ta đang **tạo lại** `n` chuỗi xoay, mỗi chuỗi `O(n)`.

Insight: viết `s + s`:
```text
s     = a b c d e
s + s = a b c d e a b c d e
            └───────┘  "cdeab" = s[2:] + s[:2] nằm sẵn ở vị trí 2
```
**Mọi** phép xoay `s[k:] + s[:k]` đều là đoạn con độ dài `n` bắt đầu tại vị trí `k` của `s + s`. Vậy:

> `goal` là phép xoay của `s` ⇔ `len(goal) == len(s)` và `goal` là **chuỗi con** của `s + s`.

Điều kiện độ dài là **bắt buộc**: `"ab"` nằm trong `"abcabc"` nhưng không phải phép xoay của `"abc"`.

In [7]:
s = "abcde"
ss = s + s
for k in range(len(s)):
    print(k, ss[k:k + len(s)], "==", rotate(s, k), "?", ss[k:k + len(s)] == rotate(s, k))
print("'ab' in 'abcabc':", "ab" in "abcabc", "nhưng độ dài khác -> không phải phép xoay")

0 abcde == abcde ? True
1 bcdea == bcdea ? True
2 cdeab == cdeab ? True
3 deabc == deabc ? True
4 eabcd == eabcd ? True
'ab' in 'abcabc': True nhưng độ dài khác -> không phải phép xoay


### 5.1. Toán tử `in` cho chuỗi con, và `str.find`

- `"cde" in "abcde"` → `True` nếu là **đoạn liên tiếp** (substring).
- `s.find(t)` trả vị trí xuất hiện đầu tiên của `t` trong `s`, hoặc `-1` nếu không có. Vị trí đó chính là **số lần shift** cần thiết!

In [8]:
print("cde" in "abcde", "ace" in "abcde")
ss = "abcdeabcde"
print("find('cdeab') =", ss.find("cdeab"), "-> cần shift 2 lần")
print("find('abced') =", ss.find("abced"))

True False
find('cdeab') = 2 -> cần shift 2 lần
find('abced') = -1


## 6. Lời giải chính — `len` bằng nhau và `goal in s + s`

```python
def rotate_string(s, goal):
    return len(s) == len(goal) and goal in s + s
```
- Vế trái chặn trường hợp độ dài khác.
- Vế phải: tìm chuỗi con. CPython cài `in` cho chuỗi bằng thuật toán tìm kiếm tối ưu (thời gian thực tế gần tuyến tính).

In [9]:
def rotate_string(s: str, goal: str) -> bool:
    return len(s) == len(goal) and goal in s + s

print(rotate_string("abcde", "cdeab"), rotate_string("abcde", "abced"))

True False


## 7. TRACE từng bước

Trace cách nối chuỗi, in mọi "cửa sổ" độ dài `n` của `s + s` và chỉ ra cửa sổ khớp:

| k | cửa sổ `ss[k:k+n]` | khớp `goal = "cdeab"`? |
|---|---|:---:|
| 0 | abcde | ✘ |
| 1 | bcdea | ✘ |
| 2 | cdeab | ✔ → True |

In [10]:
def trace_rotate_string(s: str, goal: str) -> bool:
    print(f"s={s!r}, goal={goal!r}")
    if len(s) != len(goal):
        print("  độ dài khác nhau -> False")
        return False
    ss, n = s + s, len(s)
    print(f"  s+s = {ss!r}")
    for k in range(n):
        window = ss[k:k + n]
        match = window == goal
        print(f"  k={k}: cửa sổ {window!r} {'== goal -> True' if match else '!= goal'}")
        if match:
            return True
    print("  không cửa sổ nào khớp -> False")
    return False

_ = trace_rotate_string("abcde", "cdeab")
_ = trace_rotate_string("abcde", "abced")
_ = trace_rotate_string("abc", "ab")

s='abcde', goal='cdeab'
  s+s = 'abcdeabcde'
  k=0: cửa sổ 'abcde' != goal
  k=1: cửa sổ 'bcdea' != goal
  k=2: cửa sổ 'cdeab' == goal -> True
s='abcde', goal='abced'
  s+s = 'abcdeabcde'
  k=0: cửa sổ 'abcde' != goal
  k=1: cửa sổ 'bcdea' != goal
  k=2: cửa sổ 'cdeab' != goal
  k=3: cửa sổ 'deabc' != goal
  k=4: cửa sổ 'eabcd' != goal
  không cửa sổ nào khớp -> False
s='abc', goal='ab'
  độ dài khác nhau -> False


## 8. Code nộp LeetCode — lời giải chính

In [11]:
class Solution:
    def rotateString(self, s: str, goal: str) -> bool:
        return len(s) == len(goal) and goal in s + s

sol = Solution()
assert sol.rotateString("abcde", "cdeab") is True
assert sol.rotateString("abcde", "abced") is False
print("Solution OK")

Solution OK


## 9. Cách 4 — KMP: so khớp chuỗi `O(n)` bảo đảm

Trong phỏng vấn, có thể được hỏi "không dùng `in`, cài so khớp tuyến tính". **KMP** (Knuth–Morris–Pratt) dùng **mảng tiền tố** `lps` (longest proper prefix which is also suffix):

`lps[i]` = độ dài **tiền tố thật dài nhất** của `pattern[:i+1]` đồng thời là **hậu tố** của nó.

Ví dụ `pattern = "abab"`: `lps = [0, 0, 1, 2]` (`"abab"` có `"ab"` vừa là tiền tố vừa là hậu tố).

Khi so khớp gặp ký tự sai, thay vì lùi văn bản, ta nhảy `j = lps[j-1]` trong pattern — không so lại các ký tự đã biết khớp.

In [12]:
def build_lps(p: str) -> list[int]:
    lps = [0] * len(p)
    length = 0                       # độ dài tiền tố-hậu tố hiện tại
    for i in range(1, len(p)):
        while length > 0 and p[i] != p[length]:
            length = lps[length - 1] # lùi về tiền tố-hậu tố ngắn hơn
        if p[i] == p[length]:
            length += 1
        lps[i] = length
    return lps

def kmp_find(text: str, p: str) -> int:
    lps, j = build_lps(p), 0
    for i, ch in enumerate(text):
        while j > 0 and ch != p[j]:
            j = lps[j - 1]
        if ch == p[j]:
            j += 1
            if j == len(p):
                return i - len(p) + 1   # vị trí bắt đầu khớp
    return -1

print("lps('abab')   =", build_lps("abab"))
print("lps('aabaaab')=", build_lps("aabaaab"))
print(kmp_find("abcdeabcde", "cdeab"), kmp_find("abcdeabcde", "abced"))

lps('abab')   = [0, 0, 1, 2]
lps('aabaaab')= [0, 1, 0, 1, 2, 2, 3]
2 -1


In [13]:
class Solution:
    def rotateString(self, s: str, goal: str) -> bool:
        if len(s) != len(goal):
            return False
        # mảng lps của goal
        lps, length = [0] * len(goal), 0
        for i in range(1, len(goal)):
            while length and goal[i] != goal[length]:
                length = lps[length - 1]
            if goal[i] == goal[length]:
                length += 1
            lps[i] = length
        # tìm goal trong s + s
        j = 0
        for ch in s + s:
            while j and ch != goal[j]:
                j = lps[j - 1]
            if ch == goal[j]:
                j += 1
                if j == len(goal):
                    return True
        return False

print(Solution().rotateString("abcde", "cdeab"), Solution().rotateString("abcde", "abced"))

True False


## 10. Vì sao đúng?

**Mệnh đề:** Với `len(goal) == len(s) = n`: `goal = s[k:] + s[:k]` với `0 <= k < n` ⇔ `goal` xuất hiện trong `s + s`.

- (⇒) `(s+s)[k : k+n] = s[k:] + s[:k]` (phần sau của bản `s` đầu tiên, nối với phần đầu của bản `s` thứ hai). Nên `goal` là chuỗi con.
- (⇐) Nếu `goal = (s+s)[k : k+n]` thì có thể chọn `k < n` (vì `s+s` dài `2n`, một vị trí bắt đầu `k >= n` chỉ có thể là `k = n` và cho chính `s` — cũng là phép xoay 0). Theo trên, đoạn đó là `s[k:] + s[:k]`.

**Vì sao cần kiểm tra độ dài?** Không có nó, `goal` ngắn hơn (ví dụ `"ab"`) cũng "nằm trong" `s + s` → trả nhầm `True`.

## 11. Độ phức tạp và so sánh

`n = len(s)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mô phỏng shift | `O(n²)` | `O(n)` | Đúng định nghĩa |
| 2. Thử mọi `k` slicing | `O(n²)` | `O(n)` | Gọn hơn |
| 3. `goal in s + s` | thực tế ~`O(n)` (CPython dùng thuật toán tìm chuỗi tối ưu; worst-case lý thuyết tùy phiên bản) | `O(n)` | **Khuyến nghị** |
| 4. KMP trên `s + s` | `O(n)` bảo đảm | `O(n)` | Khi phỏng vấn yêu cầu tự cài |

In [14]:
import time
n = 20000
s_big = "a" * (n - 1) + "b"
goal_big = "a" * n            # không phải phép xoay, mà khác ở rất ít ký tự
for name, f in [("mô phỏng", rotate_string_simulate), ("s+s", rotate_string),
                ("KMP", lambda a, g: Solution().rotateString(a, g))]:
    t = time.perf_counter()
    r = f(s_big, goal_big)
    print(f"{name:<10} -> {r}  ({(time.perf_counter() - t) * 1000:.1f} ms)")

mô phỏng   -> False  (20.6 ms)
s+s        -> False  (0.1 ms)
KMP        -> False  (6.5 ms)


## 12. Bộ test

1. Ví dụ đề.
2. `s == goal` (0 lần shift).
3. Độ dài khác nhau (`"abc"` vs `"ab"`) — bẫy chuỗi con.
4. Một ký tự.
5. Ký tự lặp (`"aa"`, `"abab"` vs `"baba"`).
6. Cùng tập ký tự nhưng sai thứ tự (`"abc"` vs `"acb"`) — bẫy "so sánh sorted".
7. Đối chiếu 4 cách trên dữ liệu ngẫu nhiên (bảng chữ nhỏ để nhiều trường hợp khớp).

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

check("đề 1", rotate_string("abcde", "cdeab"), True)
check("đề 2", rotate_string("abcde", "abced"), False)
check("0 lần shift", rotate_string("abc", "abc"), True)
check("độ dài khác", rotate_string("abc", "ab"), False)
check("1 ký tự", rotate_string("a", "a"), True)
check("1 ký tự khác", rotate_string("a", "b"), False)
check("lặp abab/baba", rotate_string("abab", "baba"), True)
check("cùng ký tự sai thứ tự", rotate_string("abc", "acb"), False)

PASS đề 1: True
PASS đề 2: False
PASS 0 lần shift: True
PASS độ dài khác: False
PASS 1 ký tự: True
PASS 1 ký tự khác: False
PASS lặp abab/baba: True
PASS cùng ký tự sai thứ tự: False


In [16]:
import random
random.seed(796)
kmp_solution = Solution()
for _ in range(2000):
    n = random.randint(1, 8)
    s_ = "".join(random.choice("ab") for _ in range(n))
    if random.random() < 0.5:
        g_ = rotate(s_, random.randrange(n))
    else:
        g_ = "".join(random.choice("ab") for _ in range(random.randint(1, 8)))
    expected = rotate_string_simulate(s_, g_)
    assert rotate_string_all_k(s_, g_) == expected
    assert rotate_string(s_, g_) == expected
    assert kmp_solution.rotateString(s_, g_) == expected
print("2000 cặp ngẫu nhiên: 4 cách khớp nhau")

2000 cặp ngẫu nhiên: 4 cách khớp nhau


## 13. Lỗi người mới hay gặp

### Lỗi 1: Quên kiểm tra độ dài
```python
return goal in s + s     # SAI: "ab" in "abcabc" -> True
```

### Lỗi 2: So sánh tập ký tự / sorted
```python
return sorted(s) == sorted(goal)   # SAI: "abc" vs "acb" -> True
```
Xoay giữ **thứ tự vòng** của ký tự, không chỉ tập ký tự.

### Lỗi 3: Chỉ thử `n - 1` lần và quên trường hợp 0 lần
Vòng lặp shift **trước rồi mới so** sẽ bỏ sót `s == goal` nếu chỉ chạy `n-1` lần.

### Lỗi 4: Shift sai chiều
`s[-1] + s[:-1]` là xoay **phải**. Với bài này kết quả vẫn đúng (tập các phép xoay trái và phải là như nhau), nhưng nếu đề hỏi **số lần** shift trái tối thiểu thì sai.

In [17]:
def wrong_no_len(s, goal):
    return goal in s + s

def wrong_sorted(s, goal):
    return sorted(s) == sorted(goal)

def wrong_shift_first(s, goal):
    cur = s
    for _ in range(len(s) - 1):
        cur = cur[1:] + cur[0]
        if cur == goal:
            return True
    return False

print("abc/ab   : wrong_no_len      =", wrong_no_len("abc", "ab"), "| đúng =", rotate_string("abc", "ab"))
print("abc/acb  : wrong_sorted      =", wrong_sorted("abc", "acb"), "| đúng =", rotate_string("abc", "acb"))
print("abc/abc  : wrong_shift_first =", wrong_shift_first("abc", "abc"), "| đúng =", rotate_string("abc", "abc"))

abc/ab   : wrong_no_len      = True | đúng = False
abc/acb  : wrong_sorted      = True | đúng = False
abc/abc  : wrong_shift_first = False | đúng = True


## 14. Mẹo nhận dạng pattern

- "Xoay vòng", "dịch vòng", "cyclic shift", "mảng vòng tròn" ⇒ **nhân đôi** dữ liệu: `s + s` / `nums + nums`, hoặc chỉ số `i % n`.
- "B có phải là một phép xoay / một đoạn của A" ⇒ quy về **tìm chuỗi con** (`in`, `find`, KMP, Z-function, rolling hash).
- Luôn hỏi: **độ dài có bằng nhau** không? Có cho phép 0 lần biến đổi không?

### Biến thể

1. Trả về **số lần shift trái nhỏ nhất** (hoặc `-1`): `(s + s).find(goal)` khi độ dài bằng nhau.
2. **Rotate Array (LeetCode 189)**: xoay mảng `k` bước — `nums[-k:] + nums[:-k]` hoặc đảo ngược 3 lần.
3. **Repeated Substring Pattern (459)**: `s in (s + s)[1:-1]` — cùng mẹo nhân đôi.
4. **Phép xoay nhỏ nhất theo thứ tự từ điển** (Booth's algorithm / thử mọi `k`).
5. Hai chuỗi là **xoay của nhau** khi xét **cả xoay ngược** (đảo chuỗi) — "necklace equivalence".

In [18]:
# Biến thể 1 và 3
def min_left_shifts(s, goal):
    return (s + s).find(goal) if len(s) == len(goal) else -1

def repeated_substring_pattern(s):
    return s in (s + s)[1:-1]

print(min_left_shifts("abcde", "cdeab"), min_left_shifts("abcde", "abced"), min_left_shifts("aaa", "aaa"))
print(repeated_substring_pattern("abab"), repeated_substring_pattern("aba"), repeated_substring_pattern("abcabcabc"))

2 -1 0
True False True


## 15. Bài tập tự luyện (TODO — đáp án ở cuối)

**Bài 1 (đoán output):** `rotate("python", 4)` cho chuỗi gì? `("pythonpython").find("onpyth")` bằng bao nhiêu? Gán `answer_ex1 = (chuỗi, số)`.

**Bài 2 (điền chỗ trống):** Hoàn thiện `rotate_string_exercise` theo lời giải `s + s`.

**Bài 3 (biến thể):** `rotate_right(s, k)` xoay **phải** `k` lần (ký tự cuối lên đầu), `k` có thể lớn hơn `len(s)`.

**Bài 4 (mảng):** `is_rotation_list(a, b)` kiểm tra list số `b` có phải phép xoay của list `a`. (Gợi ý: `in` không tìm "list con" được.)

**Bài 5 (mini challenge):** `smallest_rotation(s)` trả về phép xoay **nhỏ nhất theo thứ tự từ điển** của `s`.

In [19]:
answer_ex1 = None   # TODO: (str, int)

def rotate_string_exercise(s: str, goal: str) -> bool:
    # TODO 1: kiểm tra độ dài
    # TODO 2: kiểm tra goal có trong s + s
    pass

def rotate_right(s: str, k: int) -> str:
    # TODO: dùng k % len(s)
    pass

# assert rotate_string_exercise("abcde", "cdeab") == True
# assert rotate_right("abcde", 2) == "deabc"
# assert rotate_right("abcde", 7) == "deabc"

In [20]:
def is_rotation_list(a: list[int], b: list[int]) -> bool:
    # TODO
    pass

def smallest_rotation(s: str) -> str:
    # TODO
    pass

# assert is_rotation_list([1, 2, 3], [3, 1, 2]) == True
# assert is_rotation_list([1, 2, 3], [1, 3, 2]) == False
# assert smallest_rotation("cba") == "acb"

### Gợi ý 3 mức

**Bài 1** — (1) Cắt 4 ký tự đầu `"pyth"` đưa xuống cuối. (2) Kết quả bắt đầu bằng `"on"`. (3) Tìm `"onpyth"` trong `"pythonpython"`: nó bắt đầu tại chữ `o` đầu tiên.

**Bài 2** — (1) Hai điều kiện nối bằng `and`. (2) `len(s) == len(goal)`. (3) `goal in s + s`.

**Bài 3** — (1) Xoay phải `k` = xoay trái `n - k`. (2) Rút gọn `k %= n`. (3) `s[-k:] + s[:-k]` — cẩn thận `k = 0` (`s[-0:]` là cả chuỗi!) → dùng `s[n-k:] + s[:n-k]`.

**Bài 4** — (1) Nhân đôi `a + a`. (2) Thử mọi vị trí bắt đầu `k`. (3) `any((a + a)[k:k+n] == b for k in range(n))` kèm kiểm tra độ dài.

**Bài 5** — (1) Có `n` phép xoay. (2) Sinh tất cả bằng `rotate(s, k)`. (3) `min(...)` trên chuỗi so sánh theo từ điển.

---

## 16. Đáp án tham khảo

In [21]:
answer_ex1_solution = ("onpyth", 4)

def rotate_string_exercise_solution(s, goal):
    return len(s) == len(goal) and goal in s + s

def rotate_right_solution(s, k):
    n = len(s)
    k %= n
    return s[n - k:] + s[:n - k]

def is_rotation_list_solution(a, b):
    n = len(a)
    if n != len(b):
        return False
    aa = a + a
    return any(aa[k:k + n] == b for k in range(n))

def smallest_rotation_solution(s):
    return min(s[k:] + s[:k] for k in range(len(s)))

assert (rotate("python", 4), "pythonpython".find("onpyth")) == answer_ex1_solution
assert rotate_string_exercise_solution("abcde", "cdeab") is True
assert rotate_string_exercise_solution("abc", "ab") is False
assert rotate_right_solution("abcde", 2) == "deabc"
assert rotate_right_solution("abcde", 7) == "deabc"
assert rotate_right_solution("abcde", 5) == "abcde"
assert is_rotation_list_solution([1, 2, 3], [3, 1, 2]) is True
assert is_rotation_list_solution([1, 2, 3], [1, 3, 2]) is False
assert smallest_rotation_solution("cba") == "acb"
assert smallest_rotation_solution("bab") == "abb"
print("Đáp án PASS toàn bộ test!")

Đáp án PASS toàn bộ test!


## 17. Template nhớ nhanh và checklist submit

```python
class Solution:
    def rotateString(self, s: str, goal: str) -> bool:
        return len(s) == len(goal) and goal in s + s
```

Mẹo nhớ: **"xoay vòng → nhân đôi → tìm chuỗi con"**.

**Checklist trước khi submit:**
- [ ] Đã kiểm tra `len(s) == len(goal)`?
- [ ] Trường hợp `s == goal` (0 lần shift) trả `True`?
- [ ] Không dùng `sorted`/`set` (mất thứ tự)?
- [ ] Biết giải thích vì sao mọi phép xoay nằm trong `s + s`?
- [ ] Nếu bị yêu cầu không dùng `in`: cài KMP trên `s + s`.